In [34]:
import pandas as pd
import os
import re
import numpy as np

## 분개장 파일 읽기

In [35]:
def read_raw_file(raw_path):
    file_name, file_extension = os.path.splitext(os.path.basename(raw_path))

    if file_extension == ".xls":
        df = pd.read_excel(raw_path, sheet_name='Sheet1', header=0, engine='xlrd', keep_default_na=False)
    elif file_extension == ".xlsx":
        df = pd.read_excel(raw_path, sheet_name='Sheet1', header=0, engine='openpyxl', keep_default_na=False)

    return file_name, df



## 연도 / 위하고 계정 추가

In [36]:
def make_df_account():
    # wehago_account_path = "/Users/yongjae/Downloads/위하고_계정과목.json"
    # wehago_account_path = "./wehago_account.json"
    # df_account = pd.read_json(wehago_account_path)
    account_path = "./wehago_account_240813.csv" 
    df_account = pd.read_csv(account_path)
    df_account['코드'] = df_account['코드'].astype(str)
    return df_account

def get_year_from_df_raw(df):
    period = df.iloc[1, 21]
    matches = re.findall(r'\d{4}', period)
    if matches:
        return matches[0]
    else:
        print("No 4-digit number found in the period.")
        return None
    
def new_header_df(df):
    new_header = df.iloc[2] # 3번째 행을 새로운 헤더로 설정
    df = df[3:] # DataFrame에서 3번째 행을 제거
    df.columns = new_header # 새로운 헤더를 설정
    return df

def clean_column_name(df):
    df.columns = df.columns.str.replace('\n', '')
    df.columns = df.columns.str.replace('/', '')
    return df

def make_new_column(df, year):
    df['년월'] = year + "년 " + df['일자'].astype(str).str[:2].replace('^0', '', regex=True) + "월"
    df['구분자'] = str(year)[-2:] + "-" + df['일자'].astype(str).str[:2] + "-" + df['일자'].astype(str).str[-2:] + "-" + df['전표번호'].astype(str).str.zfill(5)
    cols = df.columns.tolist()
    cols.insert(0, cols.pop(cols.index('구분자')))
    cols.insert(1, cols.pop(cols.index('년월')))
    df = df.reindex(columns=cols)
    return df


def make_useful_table(df):
    wehago_account = make_df_account() # 위하고 계정 추가
    year = get_year_from_df_raw(df)

    df = new_header_df(df)
    df = clean_column_name(df)
    df = make_new_column(df, year)

    df = pd.merge(df, wehago_account[['코드', '계정과목_표준', '계정용도', '수익여부', '비용여부', '부가세여부']], left_on='계정코드', right_on="코드",  how='left')
    df.drop(['코드_x', '코드_y', '부서사원', '부서사원명', '신용카드', '신용카드명', '현장코드', '현장명', '프로젝트', '프로젝트명', '자금항목명', '관리항목'], axis=1, inplace=True)
    df = df.reset_index().rename(columns={'index': '행숫자', '계정과목': "계정과목_위하고"})

    # df.to_excel("/Users/yongjae/Downloads/df.xlsx", index=False, engine='openpyxl')
    # df.head(5)

    return df

## 차변과 대변을 1개의 행으로 다시 만들기 - 기초 데이터

In [37]:

def process_duplicate_rows(df):  
    df['차변_합계'] = np.nan
    df['차변_비중'] = np.nan
    df['차변_변환'] = df['차변']
    df['대변_합계'] = np.nan
    df['대변_비중'] = np.nan
    df['대변_대변_변환'] = df['대변_대변']
        

    if df['구분자'].duplicated().any():
        duplicated_df = df[df['구분자'].duplicated(keep=False)]

        for i, group in duplicated_df.groupby(['구분자']):

            # 분개가 N x N 인 경우
            if group['행숫자_차변'].duplicated().any() & group['행숫자_대변'].duplicated().any():
                for i, bundle in group.groupby(['구분자', '행숫자_차변']):
                    bundle['차변_합계'] = bundle['차변'].sum()
                    bundle['차변_비중'] = bundle['차변'] / bundle['차변_합계']


                    bundle['대변_합계'] = bundle['대변_대변'].sum()
                    bundle['대변_비중'] = bundle['대변_대변'] / bundle['대변_합계']
                    bundle['차변_변환'] = np.round(bundle['차변'] * bundle['대변_비중'])
                    bundle['대변_대변_변환'] = bundle['차변_변환']
                    group.update(bundle)

            else:
                if abs(group['대변_대변'].sum()) > abs(group['차변'].sum()):
                    group['차변_합계'] = group['차변'].sum()
                    group['차변_비중'] = group['차변'] / group['차변_합계']
                    group['대변_대변_변환'] = np.round(group['대변_대변'] * group['차변_비중'])

                    group['대변_합계'] = group['대변_대변'].sum()
                    group['대변_비중'] = group['대변_대변'] / group['대변_합계']
                    group['차변_변환'] = np.round(group['차변_합계'] * group['차변_비중'])
                elif abs(group['대변_대변'].sum()) < abs(group['차변'].sum()):
                    group['대변_합계'] = group['대변_대변'].sum()
                    group['대변_비중'] = group['대변_대변'] / group['대변_합계']
                    group['차변_변환'] = np.round(group['차변'] * group['대변_비중'])

                    group['차변_합계'] = group['차변'].sum()
                    group['차변_비중'] = group['차변'] / group['차변_합계']
                    group['대변_대변_변환'] = np.round(group['대변_합계'] * group['대변_비중'])



            df.update(group)
        
    return df

def make_cha_dae_table(df):
    df_cha = df[df['구분'] == '차변']
    df_dae = df[df['구분'] == '대변']

    df_cha_dae_raw = pd.merge(df_cha, df_dae[['행숫자','구분자','계정코드','계정과목_위하고', '계정과목_표준', '계정용도', '수익여부', '비용여부', '부가세여부', '대변', '거래처코드', '거래처명']], on='구분자', how='left', suffixes=('_차변', '_대변'))

    df_cha_dae = process_duplicate_rows(df_cha_dae_raw)

    # 필요 없는 컬럼 삭제
    df_cha_dae.drop(columns=['차변', '대변_대변', '대변_차변', '차변_합계', '차변_비중', '대변_합계', '대변_비중'], inplace=True)

    # df_cha_dae.to_excel("/Users/yongjae/Downloads/output_cha_dae.xlsx", index=False, engine='openpyxl')
    # df_cha_dae.head(3)

    return df_cha_dae






# 보고서 양식으로 변환

## 1. 수익 보고서

In [38]:
def make_income(df_income):
    income = df_income.pivot_table(index=['계정용도_대변', '계정과목_표준_대변'], columns='년월', values='대변_대변_변환', aggfunc='sum', margins=True, margins_name='합계')
    income = income.map(lambda x: format(int(x), ',') if pd.notnull(x) else "")
    return income

def make_income_client(df_income):
    income_client = df_income.pivot_table(index=['거래처명_대변'], columns='년월', values='대변_대변_변환', aggfunc='sum', margins=True, margins_name='합계')

    # income_client가 비어 있는지 확인
    if income_client.empty:
        raise ValueError("income_client 데이터프레임이 비어 있습니다.")

    # 열의 수가 2개 이상인지 확인
    if len(income_client.columns) < 2:
        raise ValueError("income_client 데이터프레임의 열의 수가 2개 미만입니다.")

    # 마지막 칼럼의 이름을 가져옴
    last_column = income_client.columns[-2]

    # '합계' 행을 숫자로 변환
    income_client.loc['합계'] = income_client.loc['합계'].apply(pd.to_numeric, errors='coerce')

    # 마지막 칼럼과 '합계' 칼럼을 기준으로 내림차순 정렬
    income_client = income_client.sort_values(by=[last_column, '합계'], ascending=[False, False])

    # 숫자를 포맷팅하여 문자열로 변환
    income_client = income_client.applymap(lambda x: format(int(x), ',') if pd.notnull(x) else "")
    return income_client
    

def publish_income(df_cha_dae):
    df_income = df_cha_dae[df_cha_dae['수익여부_대변'] == True]
    income = make_income(df_income)
    income_client = make_income_client(df_income)
    return income, income_client


In [39]:
df_income = df_cha_dae[df_cha_dae['수익여부_대변'] == True]

### 1-1. 수익 by 계정

In [40]:
pivot_income = df_income.pivot_table(index=['계정용도_대변', '계정과목_표준_대변'], columns='년월', values='대변_대변_변환', aggfunc='sum', margins=True, margins_name='합계')
pivot_income = pivot_income.map(lambda x: format(int(x), ',') if pd.notnull(x) else "")

pivot_income


,년월
계정용도_대변,계정과목_표준_대변


### 1-2. 수익 by 거래처

In [41]:
income_client = df_income.pivot_table(index=['거래처명_대변'], columns='년월', values='대변_대변_변환', aggfunc='sum', margins=True, margins_name='합계')

# 마지막 칼럼의 이름을 가져옴
last_column = income_client.columns[-2]

# 마지막 칼럼에 값이 있는 행만 필터링
income_client_last = income_client[income_client[last_column].notna() & (income_client[last_column] != 0)]

# '합계' 행을 숫자로 변환
income_client.loc['합계'] = income_client.loc['합계'].apply(pd.to_numeric, errors='coerce')

# 마지막 칼럼과 '합계' 칼럼을 기준으로 내림차순 정렬
income_client = income_client.sort_values(by=[last_column, '합계'], ascending=[False, False])

# 숫자를 포맷팅하여 문자열로 변환
income_client = income_client.applymap(lambda x: format(int(x), ',') if pd.notnull(x) else "")
income_client

IndexError: index -2 is out of bounds for axis 0 with size 0

### 1-3. 수익 by 거래처 in last month

In [ ]:
income_client_last = income_client_last.sort_values(by=[last_column], ascending=[ False])

# 숫자를 포맷팅하여 문자열로 변환
income_client_last = income_client_last.applymap(lambda x: format(int(x), ',') if pd.notnull(x) else "")

income_client_last

/var/folders/jm/6pbq_zts6k12d5j3w0c7pr6h0000gn/T/ipykernel_2061/249015130.py:4: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  income_client_last = income_client_last.applymap(lambda x: format(int(x), ',') if pd.notnull(x) else "")


년월,2024년 1월,2024년 2월,2024년 3월,2024년 4월,2024년 5월,2024년 6월,합계
거래처명_대변,,,,,,,
합계,"155,629,000","98,943,548","49,630,218","45,911,420","53,656,909","54,232,049","458,003,144"
현영매출,,,,"26,295,874","10,155,000","12,619,140","49,070,014"
케이씨로보테크(주),,,,,,"9,090,909","9,090,909"
우경인더스트리,,,,,,"6,792,000","6,792,000"
한국물류설비제작소(주),,,,,,"6,000,000","6,000,000"
주식회사 에스씨엘헬스케어,,,,,,"5,500,000","5,500,000"
구래,,,,,,"5,459,000","5,459,000"
삼성세무서,,,,,,"3,936,000","3,936,000"
(유)샤이앤브레이브,,,,,,"2,500,000","2,500,000"


## 2. 비용 보고서

In [ ]:
def make_cost(df_cost):
    cost = df_cost.pivot_table(index=['계정용도_차변', '계정과목_표준_차변'], columns='년월', values='차변_변환', aggfunc='sum', margins=True, margins_name='합계')
    cost = cost.map(lambda x: format(int(x), ',') if pd.notnull(x) else "")
    return cost

def make_cost_client(df_cost):

    cost_client = df_cost.pivot_table(index=['거래처명_차변'], columns='년월', values='차변_변환', aggfunc='sum', margins=True, margins_name='합계')

    # 마지막 칼럼의 이름을 가져옴
    last_column = cost_client.columns[-2]
    # '합계' 행을 숫자로 변환
    cost_client.loc['합계'] = cost_client.loc['합계'].apply(pd.to_numeric, errors='coerce')

    # 마지막 칼럼과 '합계' 칼럼을 기준으로 내림차순 정렬
    cost_client = cost_client.sort_values(by=[last_column, '합계'], ascending=[False, False])

    # 숫자를 포맷팅하여 문자열로 변환
    cost_client = cost_client.applymap(lambda x: format(int(x), ',') if pd.notnull(x) else "")

    return cost_client


def publish_cost(df_cha_dae):
    df_cost = df_cha_dae[df_cha_dae['비용여부_차변'] == True]
    cost = make_cost(df_cost)
    cost_client = make_cost_client(df_cost)
    return cost, cost_client

In [ ]:
df_expense = df_cha_dae[(df_cha_dae['수익여부_대변'] != '수익') & (~df_cha_dae['R계정과목_차변'].isin(['단순이체', '미지급금', '부가세대급금', '부가세예수금', '보통예금', '가지급금', '미수수익', '미지급비용','비품', '전도금', '예수금', '외상매출금', '선수금', '선납세금', '선급금', '미지급세금']))]

### 2-1. 비용 by 계정

In [ ]:
expense_account = df_expense.pivot_table(index=['계정용도_차변', 'R계정과목_차변'], columns='년월', values='차변_변환', aggfunc='sum', margins=True, margins_name='합계')
expense_account = expense_account.map(lambda x: format(int(x), ',') if pd.notnull(x) else "")

expense_account

년월                  2024년 1월    2024년 2월    2024년 3월    2024년 4월    2024년 5월  \
계정용도_차변 R계정과목_차변                                                               
사업      건물관리비        846,630     872,960     858,560     764,120     659,170   
        광고선전비     13,148,753   5,310,661   4,233,153   4,162,129   2,903,174   
        대손상각비                                                                  
        도서인쇄비        796,464     224,682     532,091     340,605               
        보험료          116,850     114,510     116,590     116,590     844,190   
        복리후생비      3,621,922  13,596,101   3,108,036   4,070,928   4,039,075   
        상여금                    5,900,000   5,999,999   5,000,000   5,000,000   
        세금과공과금       358,350     400,760     359,050     359,050     358,850   
        소모품비       5,735,715   3,081,670   3,353,326   2,481,699   4,017,554   
        여비교통비        534,800     241,400     536,500   2,113,850     350,000   
        운반비          992,538     460,981     717,634     678,109     382,418   
        잡손실                                               56,000      32,000   
        접대비        1,986,980     575,700   1,744,800   2,217,600     348,800   
        지급수수료     34,268,769   2,174,261  36,576,460   9,106,177   8,588,965   
        지급임차료      2,027,208   1,769,288   1,769,288   1,769,288   1,769,288   
        직원급여      16,729,000  16,295,409  17,566,658  34,241,050  25,946,087   
        차량유지비        435,100     200,000     613,000     356,000     375,700   
        통신비          136,141     156,968     129,166     134,632     140,585   
        퇴직급여       1,071,330   1,071,330   1,071,330   1,071,330   1,071,330   
합계                82,806,550  52,446,682  79,285,643  69,039,157  56,827,185   

년월                   2024년 6월           합계  
계정용도_차변 R계정과목_차변                            
사업      건물관리비         631,110    4,632,550  
        광고선전비       1,580,872   31,338,742  
        대손상각비      99,000,000   99,000,000  
        도서인쇄비         335,883    2,229,725  
        보험료           667,880    1,976,610  
        복리후생비       4,735,499   33,171,561  
        상여금         5,000,000   26,899,999  
        세금과공과금        358,850    2,194,910  
        소모품비        4,946,405   23,616,368  
        여비교통비         404,800    4,181,350  
        운반비           277,500    3,509,180  
        잡손실                         88,000  
        접대비         2,831,180    9,705,060  
        지급수수료       7,999,845   98,714,477  
        지급임차료       1,769,288   10,873,648  
        직원급여       11,034,600  121,812,806  
        차량유지비       2,415,620    4,395,420  
        통신비           139,390      836,882  
        퇴직급여                     5,356,650  
합계                144,128,722  484,533,939

### 2-2. 비용 by 거래처

In [ ]:
expense_client = df_expense.pivot_table(index=['거래처명_차변'], columns='년월', values='차변_변환', aggfunc='sum', margins=True, margins_name='합계')

# 마지막 칼럼의 이름을 가져옴
last_column = expense_client.columns[-2]
# '합계' 행을 숫자로 변환
expense_client.loc['합계'] = expense_client.loc['합계'].apply(pd.to_numeric, errors='coerce')

# 마지막 칼럼과 '합계' 칼럼을 기준으로 내림차순 정렬
expense_client = expense_client.sort_values(by=[last_column, '합계'], ascending=[False, False])

# 숫자를 포맷팅하여 문자열로 변환
expense_client = expense_client.applymap(lambda x: format(int(x), ',') if pd.notnull(x) else "")

expense_client

/var/folders/jm/6pbq_zts6k12d5j3w0c7pr6h0000gn/T/ipykernel_2061/1701854884.py:12: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  expense_client = expense_client.applymap(lambda x: format(int(x), ',') if pd.notnull(x) else "")


년월,2024년 1월,2024년 2월,2024년 3월,2024년 4월,2024년 5월,2024년 6월,합계
거래처명_차변,,,,,,,
합계,"82,806,550","52,446,682","79,285,643","69,039,157","56,827,185","144,128,722","484,533,939"
,"21,303,020","24,593,451","25,630,817","44,355,168","35,630,654","116,657,258","268,170,368"
정진세림회계법인,,,,,"3,000,000","3,000,000","6,000,000"
원스탑세무행정컨설팅,"269,520",,,,,"1,999,300","2,268,820"
네이버 주식회사,"5,148,753","5,310,661","4,233,153","4,162,129","2,903,174","1,580,872","23,338,742"
...,...,...,...,...,...,...,...
미금프라자운영위원회,,"1,000",,,,,"1,000"
주식회사 드림글로벌,,,,"1,000",,,"1,000"
홍대와이즈파킹,"1,000",,,,,,"1,000"


## 3. 부가세 보고서

In [ ]:
def publish_vat(df_cha_dae):

    df_vat = df_cha_dae[(df_cha_dae['부가세여부_차변'] == True) | (df_cha_dae['부가세여부_대변'] == True)]

    columns_to_keep = ['계정용도_대변', '계정과목_표준_차변', '계정과목_표준_대변', '년월', '차변_변환', '대변_대변_변환']
    df_vat_simple = df_vat[columns_to_keep]


    df_vat_cash_out = df_vat_simple[df_vat_simple['계정과목_표준_대변'] == '부가세예수금']
    df_vat_cash_out['부가세예정'] = df_vat_cash_out['대변_대변_변환'] * -1
    df_vat_cash_out['계정과목_표준_차변'] = ""
    df_vat_cash_in = df_vat_simple[df_vat_simple['계정과목_표준_차변'] == '부가세대급금']
    df_vat_cash_in['부가세예정'] = df_vat_cash_in['차변_변환']
    df_vat_cash_in['계정과목_표준_대변'] = ""

    df_vat_combined = pd.concat([df_vat_cash_out, df_vat_cash_in], ignore_index=True)

    vat = df_vat_combined.pivot_table(index=['계정과목_표준_차변', '계정과목_표준_대변'], columns='년월', values=['부가세예정'], aggfunc='sum', margins=True, margins_name='합계')
    vat = vat.map(lambda x: format(int(x), ',') if pd.notnull(x) else "")
    return vat


In [ ]:
def delete_worthless_vat_values(df):
    df.loc[df['계정코드_차변'] == '255', '대변_대변'] = np.nan # 255 부가세예수금
    df.loc[df['계정코드_대변'] == '255', '차변'] = np.nan # 255 부가세예수금
    df.loc[df['계정코드_차변'] == '135', '대변_대변'] = np.nan # 135 부가세대급금
    df.loc[df['계정코드_대변'] == '135', '차변'] = np.nan # 135 부가세대급금
    return df


df_vat = df_cha_dae[(df_cha_dae['부가세여부_차변'] == 'TRUE') | (df_cha_dae['부가세여부_대변'] == 'TRUE')]
# df_vat = delete_worthless_vat_values(df_vat)


columns_to_keep = ['계정용도_대변', 'R계정과목_차변', 'R계정과목_대변', '년월', '차변_변환', '대변_대변_변환']
df_vat_simple = df_vat[columns_to_keep]


df_vat_cash_out = df_vat_simple[df_vat_simple['R계정과목_대변'] == '부가세예수금']
df_vat_cash_out['부가세예정'] = df_vat_cash_out['대변_대변_변환'] * -1
df_vat_cash_out['R계정과목_차변'] = ""
df_vat_cash_in = df_vat_simple[df_vat_simple['R계정과목_차변'] == '부가세대급금']
df_vat_cash_in['부가세예정'] = df_vat_cash_in['차변_변환']
df_vat_cash_in['R계정과목_대변'] = ""

df_vat_combined = pd.concat([df_vat_cash_out, df_vat_cash_in], ignore_index=True)

pivot_vat = df_vat_combined.pivot_table(index=['R계정과목_차변', 'R계정과목_대변'], columns='년월', values=['부가세예정'], aggfunc='sum', margins=True, margins_name='합계')
pivot_vat = pivot_vat.map(lambda x: format(int(x), ',') if pd.notnull(x) else "")
pivot_vat


/var/folders/jm/6pbq_zts6k12d5j3w0c7pr6h0000gn/T/ipykernel_2061/2084278107.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_vat_cash_out['부가세예정'] = df_vat_cash_out['대변_대변_변환'] * -1
/var/folders/jm/6pbq_zts6k12d5j3w0c7pr6h0000gn/T/ipykernel_2061/2084278107.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_vat_cash_out['R계정과목_차변'] = ""
/var/folders/jm/6pbq_zts6k12d5j3w0c7pr6h0000gn/T/ipykernel_2061/2084278107.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a 

부가세예정                                      \
년월                    2024년 1월    2024년 2월    2024년 3월    2024년 4월   
R계정과목_차변 R계정과목_대변                                                    
         부가세예수금    -15,562,900  -9,894,352  -4,963,022  -4,591,140   
부가세대급금               5,222,116   1,130,385   4,575,820     949,562   
합계                 -10,340,784  -8,763,967    -387,202  -3,641,578   

                                                        
년월                   2024년 5월    2024년 6월           합계  
R계정과목_차변 R계정과목_대변                                       
         부가세예수금    -5,365,691  -5,423,201  -45,800,306  
부가세대급금              1,617,906   1,220,216   14,716,005  
합계                 -3,747,785  -4,202,985  -31,084,301

## 4. 현금 흐름 보고서

In [45]:
def publish_cash_in(df_cha_dae):
    df_cash_in = df_cha_dae[df_cha_dae['계정과목_위하고_차변'] == '보통예금']

    cash_in = df_cash_in.pivot_table(index=['거래처명_대변', '거래처명_차변'], columns='년월', values='대변_대변_변환', aggfunc='sum', margins=True, margins_name='합계')

    # 마지막 칼럼의 이름을 가져옴
    last_column = cash_in.columns[-2]

    # 마지막 칼럼과 '합계' 칼럼을 기준으로 내림차순 정렬
    cash_in = cash_in.sort_values(by=[last_column, '합계'], ascending=[False, False])

    # 숫자를 포맷팅하여 문자열로 변환
    cash_in = cash_in.applymap(lambda x: format(int(x), ',') if pd.notnull(x) else "")

    return cash_in

def publish_cash_out(df_cha_dae):
    df_cash_out = df_cha_dae[df_cha_dae['계정과목_위하고_대변'] == '보통예금']

    cash_out = df_cash_out.pivot_table(index=['거래처명_차변', '거래처명_대변'], columns='년월', values='대변_대변_변환', aggfunc='sum', margins=True, margins_name='합계')

    # 마지막 칼럼의 이름을 가져옴
    last_column = cash_out.columns[-2]

    # 마지막 칼럼과 '합계' 칼럼을 기준으로 내림차순 정렬
    cash_out = cash_out.sort_values(by=[last_column, '합계'], ascending=[False, False])
    # 숫자를 포맷팅하여 문자열로 변환
    cash_out = cash_out.applymap(lambda x: format(int(x), ',') if pd.notnull(x) else "")

    return cash_out
    


def publish_cash(df_cha_dae):
    cash_in = publish_cash_in(df_cha_dae)
    cash_out = publish_cash_out(df_cha_dae) 

    return cash_in, cash_out

### 4-1. 현금 유입

In [48]:
df_cash_in = df_cha_dae[df_cha_dae['계정과목_위하고_차변'] == '보통예금']

cash_in = df_cash_in.pivot_table(index=['거래처명_대변', '거래처명_차변'], columns='년월', values='대변_대변_변환', aggfunc='sum', margins=True, margins_name='합계')

# 마지막 칼럼의 이름을 가져옴
last_column = cash_in.columns[-2]


# 최근월에 대한 데이터만 추출
cash_in_last_month = cash_in[(cash_in[last_column] != 0) & (cash_in[last_column].notna())]
cash_in_last_month = cash_in_last_month.sort_values(by=[last_column], ascending=[False])
cash_in_last_month = cash_in_last_month.applymap(lambda x: format(int(x), ',') if pd.notnull(x) else "")


# 마지막 칼럼과 '합계' 칼럼을 기준으로 내림차순 정렬
cash_in = cash_in.sort_values(by=[last_column, '합계'], ascending=[False, False])


# 숫자를 포맷팅하여 문자열로 변환
cash_in = cash_in.applymap(lambda x: format(int(x), ',') if pd.notnull(x) else "")

cash_in

/var/folders/jm/6pbq_zts6k12d5j3w0c7pr6h0000gn/T/ipykernel_3668/2322457958.py:12: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  cash_in_last_month = cash_in_last_month.applymap(lambda x: format(int(x), ',') if pd.notnull(x) else "")
/var/folders/jm/6pbq_zts6k12d5j3w0c7pr6h0000gn/T/ipykernel_3668/2322457958.py:20: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  cash_in = cash_in.applymap(lambda x: format(int(x), ',') if pd.notnull(x) else "")


,년월,2025년 1월,2025년 2월,2025년 3월,2025년 4월,2025년 5월,2025년 6월,합계
거래처명_대변,거래처명_차변,,,,,,,
합계,,"133,660,813","179,469,855","166,777,782","138,172,815","154,100,012","159,950,021","932,131,298"
위더스제약 (주),국민은행,"121,118,221","165,312,125","153,253,341","126,335,797","140,052,351","145,052,216","851,124,051"
에이치피씨,국민은행,"7,889,249","9,505,572","8,352,900","7,955,347","8,576,206","9,607,636","51,886,910"
(주)케이엠에스컴퍼니,국민은행,"2,837,715","2,611,830","3,080,360","2,233,642","3,060,151","3,096,921","16,920,619"
주식회사 제이엔케이,국민은행,"1,815,628","2,040,328","2,091,181","1,648,029","2,411,304","2,186,845","12,193,315"
,국민은행,,,,,,"6,403","6,403"


### 4-2. 현금 유출

In [47]:
df_cash_out = df_cha_dae[df_cha_dae['계정과목_위하고_대변'] == '보통예금']

cash_out = df_cash_out.pivot_table(index=['거래처명_차변', '거래처명_대변'], columns='년월', values='대변_대변_변환', aggfunc='sum', margins=True, margins_name='합계')

# 마지막 칼럼의 이름을 가져옴
last_column = cash_out.columns[-2]

# 최근월에 대한 데이터만 추출
cash_out_last_month = cash_out[(cash_out[last_column] != 0) & (cash_out[last_column].notna())]
cash_out_last_month = cash_out_last_month.sort_values(by=[last_column], ascending=[False])
cash_out_last_month = cash_out_last_month.applymap(lambda x: format(int(x), ',') if pd.notnull(x) else "")



# 마지막 칼럼과 '합계' 칼럼을 기준으로 내림차순 정렬
cash_out = cash_out.sort_values(by=[last_column, '합계'], ascending=[False, False])
# 숫자를 포맷팅하여 문자열로 변환
cash_out = cash_out.applymap(lambda x: format(int(x), ',') if pd.notnull(x) else "")

cash_out




/var/folders/jm/6pbq_zts6k12d5j3w0c7pr6h0000gn/T/ipykernel_3668/1953818688.py:11: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  cash_out_last_month = cash_out_last_month.applymap(lambda x: format(int(x), ',') if pd.notnull(x) else "")
/var/folders/jm/6pbq_zts6k12d5j3w0c7pr6h0000gn/T/ipykernel_3668/1953818688.py:18: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  cash_out = cash_out.applymap(lambda x: format(int(x), ',') if pd.notnull(x) else "")


,년월,2025년 1월,2025년 2월,2025년 3월,2025년 4월,2025년 5월,2025년 6월,합계
거래처명_차변,거래처명_대변,,,,,,,
합계,,"134,925,169","163,919,205","172,079,841","144,705,905","146,648,510","151,316,078","913,594,708"
주식회사 제이드수컴퍼니,국민은행,"44,026,194","75,862,754","73,592,179","60,258,219","68,786,909","69,884,133","392,410,388"
(주)스파이두프로덕션,국민은행,,,"14,300,000","14,300,000","14,300,000","14,300,000","57,200,000"
서강,국민은행,"12,139,042","12,624,756","12,655,643","11,195,817","12,869,526","13,864,972","75,349,756"
주식회사 리아이티,국민은행,"6,872,796","7,683,074","7,764,427","7,900,000","8,873,759","10,008,938","49,102,994"
（주）국민은행,국민은행,"5,408,980","5,089,220","3,355,280","5,439,350","8,628,348","9,696,280","37,617,458"
급여,국민은행,,"9,974,206","9,742,436","9,653,926","9,653,926","9,680,226","48,704,720"
주식회사 에스제이컴퍼니,국민은행,"8,164,136","10,737,964","9,858,563","6,514,378","7,622,082","8,277,609","51,174,732"
에이치피씨,국민은행,"7,555,779","8,305,232","7,702,581","6,004,187","6,444,480","6,679,342","42,691,601"


## 엑셀로 다운로드

In [ ]:

download_path = "/Users/yongjae/Downloads/output_final.xlsx"
with pd.ExcelWriter(download_path, engine='openpyxl') as writer:
    df_cha_dae.to_excel(writer, sheet_name='분개장', index=False)
    pivot_income.to_excel(writer, sheet_name='수익')
    income_client.to_excel(writer, sheet_name='수익by거래처')
    income_client_last.to_excel(writer, sheet_name='수익by거래처in최근월')
    expense_account.to_excel(writer, sheet_name='비용')
    expense_client.to_excel(writer, sheet_name='비용by거래처')
    cash_in.to_excel(writer, sheet_name='현금유입')
    cash_in_last_month.to_excel(writer, sheet_name='현금유입in최근월')
    cash_out.to_excel(writer, sheet_name='현금유출')
    cash_out_last_month.to_excel(writer, sheet_name='현금유출in최근월')
    pivot_vat.to_excel(writer, sheet_name='부가세')




In [42]:
def make_basic_table(raw_path):

    file_name, df = read_raw_file(raw_path)
    df = make_useful_table(df)
    df_cha_dae = make_cha_dae_table(df)

    # df_cha_dae.to_excel("/Us ers/yongjae/Downloads/output_cha_dae.xlsx", index=False, engine='openpyxl')
   
    return df_cha_dae, file_name

def publish_report(df_cha_dae, file_name):
    print("Publishing report...", df_cha_dae.shape, file_name)
    # income, income_client = publish_income(df_cha_dae)
    cost, cost_client = publish_cost(df_cha_dae)
    cash_in, cash_out = publish_cash(df_cha_dae)
    vat = publish_vat(df_cha_dae)



    # download_path = "/Users/yongjae/Downloads/output_final.xlsx"
    download_path = f"./uploads/{file_name}_converted.xlsx"
    with pd.ExcelWriter(download_path, engine='openpyxl') as writer:
        df_cha_dae.to_excel(writer, sheet_name='분개장', index=False)
        income.to_excel(writer, sheet_name='수익')
        income_client.to_excel(writer, sheet_name='수익by거래처')
        cost.to_excel(writer, sheet_name='비용')
        cost_client.to_excel(writer, sheet_name='비용by거래처')
        cash_in.to_excel(writer, sheet_name='현금유입')
        cash_out.to_excel(writer, sheet_name='현금유출')
        vat.to_excel(writer, sheet_name='부가세')
    return 


def convert_bungaejang(raw_path):
    df_cha_dae, file_name = make_basic_table(raw_path)
    publish_report(df_cha_dae, file_name)

    return


# raw_path = "/Users/yongjae/Downloads/라인감평_분개장_24년_1월_7월.xls"
raw_path = "/Users/yongjae/Downloads/SABK0113_03.xls"
convert_bungaejang(raw_path)


/var/folders/jm/6pbq_zts6k12d5j3w0c7pr6h0000gn/T/ipykernel_3668/2665519944.py:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['년월'] = year + "년 " + df['일자'].astype(str).str[:2].replace('^0', '', regex=True) + "월"
/var/folders/jm/6pbq_zts6k12d5j3w0c7pr6h0000gn/T/ipykernel_3668/2665519944.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['구분자'] = str(year)[-2:] + "-" + df['일자'].astype(str).str[:2] + "-" + df['일자'].astype(str).str[-2:] + "-" + df['전표번호'].astype(str).str.zfill(5)
/var/folders/jm/6pbq

Publishing report... (1695, 29) SABK0113_03


NameError: name 'publish_cash' is not defined

In [ ]:
raw_path = "/Users/yongjae/Downloads/SABK0113_03.xls"


file_name, df = read_raw_file(raw_path)
df = make_useful_table(df)
df_cha_dae = make_cha_dae_table(df)
df_cha_dae

/var/folders/jm/6pbq_zts6k12d5j3w0c7pr6h0000gn/T/ipykernel_3668/2665519944.py:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['년월'] = year + "년 " + df['일자'].astype(str).str[:2].replace('^0', '', regex=True) + "월"
/var/folders/jm/6pbq_zts6k12d5j3w0c7pr6h0000gn/T/ipykernel_3668/2665519944.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['구분자'] = str(year)[-2:] + "-" + df['일자'].astype(str).str[:2] + "-" + df['일자'].astype(str).str[-2:] + "-" + df['전표번호'].astype(str).str.zfill(5)
/var/folders/jm/6pbq

In [43]:
df_cha_dae
df_cha_dae.to_excel("/Users/yongjae/Downloads/output_cha_dae.xlsx", index=False, engine='openpyxl')

In [ ]:
df
df.to_excel("/Users/yongjae/Downloads/output_df.xlsx", index=False, engine='openpyxl')

In [46]:
cash_in = publish_cash_in(df_cha_dae)
cash_in.to_excel("/Users/yongjae/Downloads/output_cash_in.xlsx", index=True, engine='openpyxl')

/var/folders/jm/6pbq_zts6k12d5j3w0c7pr6h0000gn/T/ipykernel_3668/4039881813.py:13: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  cash_in = cash_in.applymap(lambda x: format(int(x), ',') if pd.notnull(x) else "")
